## Type-checking helpers

The course libraries we use here (`gitsource`, `minsearch`, and `toyaikit`) work at runtime, but they do not give Pyright enough type information.

To keep the notebook readable and type-checkable, we define a few small `Protocol`s up front. A `Protocol` is a plain-language contract: *this object must have these methods or attributes*.

We use them to tell Pyright:

* `CourseSearchIndex` can `fit(...)` documents and `search(...)` them.
* `RAGAgent` has a `rag(...)` method that returns text.
* `ToolRegistry` can register a search function as a tool.
* `ResponsesRunner` can run one tool-using LLM loop and returns something with a `cost`.

These helpers do not change runtime behavior. They only make the notebook explicit about the shapes it expects from untyped third-party objects.


In [1]:
from collections.abc import Callable, Sequence
from typing import Protocol, cast

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

from lib.types import LessonChunk, LessonDocument


type SearchDocument = LessonDocument | LessonChunk


class CourseSearchIndex(Protocol):
    def fit(self, docs: Sequence[SearchDocument]) -> "CourseSearchIndex": ...

    def search(
        self,
        query: str,
        filter_dict: dict[str, object] | None = None,
        boost_dict: dict[str, float] | None = None,
        num_results: int = 10,
        output_ids: bool = False,
    ) -> list[SearchDocument]: ...


class RAGAgent(Protocol):
    def rag(self, query: str) -> str: ...


class ToolRegistry(Protocol):
    def add_tool(
        self,
        function: Callable[[str], list[SearchDocument]],
    ) -> None: ...


class RunnerLoopResult(Protocol):
    cost: object


class ResponsesRunner(Protocol):
    def loop(
        self,
        prompt: str,
        callback: object | None = None,
    ) -> RunnerLoopResult: ...

## Q1. How many lesson pages

First, we pull the lesson pages straight from the course repository. We use commit `8c1834d` so everyone works with the exact same data.

How many lesson pages are in the dataset?

* ❌ 24
* ✅ 72
* ❌ 240
* ❌ 720

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = cast(list[LessonDocument], [file.parse() for file in reader.read()])

print(len(documents))

72


## Q2. Indexing and searching

Index the documents with `minsearch` using `content` as a text field and `filename` as a keyword field.

Then search with this query:

> How does the agentic loop keep calling the model until it stops?

What's the `filename` of the first result?

* ❌ 01-agentic-rag/lessons/03-rag.md
* ✅ 01-agentic-rag/lessons/14-agentic-loop.md
* ❌ 04-evaluation/lessons/13-llm-as-judge.md
* ❌ 06-best-practices/lessons/02-hybrid-search.md

In [3]:
document_index = cast(
    CourseSearchIndex,
    Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    ),
)

document_index.fit(documents)

search_results = document_index.search(
    query="How does the agentic loop keep calling the model until it stops?"
)

print(search_results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG

Now we build a RAG assistant on top of this data.

Build a RAG over the index from Q2 and answer this query:

> How does the agentic loop keep calling the model until it stops?

Use `OPENAI_MODEL_NAME`, which defaults to `gpt-5.4-mini`.

How many input tokens did we send to the model for this request?

* ❌ 700
* ✅ 7000
* ❌ 70000
* ❌ 700000

In [4]:
import os

from homework_rag_helper import RAGBase
from openai import OpenAI


openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

agent = cast(
    RAGAgent,
    RAGBase(
        index=document_index,
        llm_client=openai_client,
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini"),
    ),
)

# I decided to keep the usage shown as part of the rag function call
# instead of complicating the return type.
response = agent.rag("How does the agentic loop keep calling the model until it stops?")

print(response)

ModelCallMetrics(model='gpt-5.4-mini-2026-03-17', input_tokens=7136, cached_input_tokens=0, output_tokens=83, reasoning_output_tokens=0, duration_seconds=4.592123800000991, price=UsagePrice(input_cost_usd=0.005352, output_cost_usd=0.0003735), completed_at=datetime.datetime(2026, 7, 13, 13, 12, 3, 233054, tzinfo=datetime.timezone.utc))
It keeps calling the model inside a `while True` loop. After each response, it checks whether the model returned any `function_call` items:

- if yes, it runs the tool, appends the tool output to `messages`, and loops again
- if no, it breaks out of the loop

So the stop condition is: **no function calls in the latest response**.


## Q4. Chunking

Long lesson pages make retrieval less precise, so we split them into overlapping chunks.

Use `chunk_documents` with `size=2000` and `step=1000`.

How many chunks do you get?

* ❌ 70
* ✅ 295
* ❌ 1100
* ❌ 4500

In [5]:
chunks = cast(
    list[LessonChunk],
    chunk_documents(
        cast(list[dict[str, str]], documents),
        size=2000,
        step=1000,
    ),
)

len(chunks)

295

## Q5. RAG with chunking

Chunking makes each request smaller because we send a smaller context to the LLM.

Index the chunks from Q4 with `content` as a text field and `filename` as a keyword field, then answer the same query again and compare the input tokens with Q3.

How many fewer input tokens does the chunked version send?

* ❌ about the same
* ✅ 3× fewer
* ❌ 10× fewer
* ❌ 30× fewer

In [6]:
chunk_index = cast(
    CourseSearchIndex,
    Index(
        text_fields=["content"],
        keyword_fields=["filename"],
    ),
)

chunk_index.fit(chunks)


openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

agent = cast(
    RAGAgent,
    RAGBase(
        index=chunk_index,
        llm_client=openai_client,
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini"),
    ),
)

response = agent.rag("How does the agentic loop keep calling the model until it stops?")

ModelCallMetrics(model='gpt-5.4-mini-2026-03-17', input_tokens=2319, cached_input_tokens=0, output_tokens=120, reasoning_output_tokens=0, duration_seconds=1.7703244000003906, price=UsagePrice(input_cost_usd=0.00173925, output_cost_usd=0.00054), completed_at=datetime.datetime(2026, 7, 13, 13, 12, 5, 99580, tzinfo=datetime.timezone.utc))


## Q6. Turning it into an agent

So far search runs once with the exact query. Now we make it agentic by giving the LLM a `search` tool and letting it decide when and what to search.

Use these instructions for the agent:

> You're a course teaching assistant.
> Answer the student's question using the search tool.
> Make multiple searches with different keywords before answering.

Ask it:

> How does the agentic loop work, and how is it different from plain RAG?

How many times did the agent call `search`?

* ❌ 0
* ✅ 4
* ❌ 10
* ❌ 20

In [7]:
from toyaikit.chat.interface import ChatInterface, IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback, OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools


instructions = """
You're a course teaching assistant.
Answer the student's question using the search tool.
Make multiple searches with different keywords before answering.
""".strip()


def search(query: str) -> list[SearchDocument]:
    """
    Search the FAQ database for entries matching the given query.
    """
    boost_dict = {"filename": 2.0, "content": 1.0}
    # filter_dict = {'course': self.course}

    return chunk_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        # filter_dict=filter_dict
    )


tools = Tools()
cast(ToolRegistry, tools).add_tool(search)

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(cast(ChatInterface, chat_interface))

runner = cast(
    ResponsesRunner,
    OpenAIResponsesRunner(
        tools=tools,
        developer_prompt=instructions,
        chat_interface=cast(ChatInterface, chat_interface),
        llm_client=OpenAIClient(model=os.getenv("OPENAI_MODEL_NAME", "gpt-5.4-mini")),
    ),
)

result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print(result.cost)

-> Response received


-> Response received


CostInfo(input_cost=Decimal('0.00625725'), output_cost=Decimal('0.001809'), total_cost=Decimal('0.00806625'))
